In [0]:
import dlt 
from pyspark.sql.functions import *

In [0]:
expec_coaches = {
    "rule1" : "code is not null",
    "rule2" : "current is True"
}

In [0]:
expec_nocs = {
    "rule1" : "code is not null"
}

In [0]:
expec_events = {
    "rule1" : "event is not null"
}

In [0]:
@dlt.table

def source_coaches():
    df= spark.readStream.table("olympics.silver.coaches")
    return df

---------------------------------------------------------------------------
Py4JError                                 Traceback (most recent call last)
File <command-7324433957619449>, line 1
----> 1 @dlt.table
      2 
      3 def source_coaches():
      4     df= spark.readStream.table("olympics.silver.coaches")
      5     return df

File /databricks/spark/python/pysparkpipelines/api.py:519, in table(query_function, name, comment, spark_conf, table_properties, partition_cols, path, schema, temporary, cluster_by, cluster_by_auto, auto_ttl, row_filter, private, replace_where)
    456 def table(
    457     query_function: Optional[Union[DatasetOrExpectationDecoratorResult, QueryFunction]] = None,
    458     name: Optional[str] = None,
   (...)
    477     DatasetOrExpectationDecoratorResult,
    478 ]:
    479     """
    480     (Return a) decorator to define a table in the pipeline and mark a function as the table's query
    481     function.
   (...)
    517         Pyspark Colum

In [0]:
@dlt.view

def view_coaches():
    df = spark.readStream.table("LIVE.source_coaches")
    df = df.fillna("Unknown")
    return df

---------------------------------------------------------------------------
Py4JError                                 Traceback (most recent call last)
File <command-7324433957619451>, line 1
----> 1 @dlt.view
      2 
      3 def view_coaches():
      4     df = spark.readStream.table("LIVE.source_coaches")
      5     df = df.fillna("Unknown")

File /databricks/spark/python/pysparkpipelines/api.py:826, in view(query_function, name, comment, spark_conf)
    797 def view(
    798     query_function: Optional[Union[QueryFunction, DatasetOrExpectationDecoratorResult]] = None,
    799     name: Optional[str] = None,
   (...)
    807     DatasetOrExpectationDecoratorResult,
    808 ]:
    809     """
    810     (Return a) decorator to define a view in the pipeline and mark a function as the view's query
    811     function.
   (...)
    824         confs set for the pipeline or on the cluster.
    825     """
--> 826     return _view(
    827         decorator_name="view",
    828       

In [0]:
@dlt.table

@dlt.expect_all_or_drop(expec_coaches)
def coaches():
    df = spark.readStream.table("LIVE.view_coaches")
    return df

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7324433957619452>, line 3
      1 @dlt.table
      2 
----> 3 @dlt.expect_all_or_drop(expec_coaches)
      4 def coaches():
      5     df = spark.readStream.table("LIVE.view_coaches")
      6     return df

NameError: name 'expec_coaches' is not defined

# NOCS DLT Pipeline

In [0]:
@dlt.view

def view_nocs():
    df = spark.readStream.table("olympics.silver.nocs")
    return df

In [0]:
@dlt.table
@dlt.expect_all_or_drop(expec_nocs)
def nocs():
    df = spark.readStream.table("LIVE.view_nocs")
    return df

In [0]:
@dlt.view

def view_events():
    df = spark.readStream.table("olympics.silver.events")
    return df
    
@dlt.table
@dlt.expect_all_or_drop(expec_events)
def events():
    df = spark.readStream.table("LIVE.view_events")
    return df

---------------------------------------------------------------------------
Py4JError                                 Traceback (most recent call last)
File <command-7324433957619454>, line 1
----> 1 @dlt.view
      2 
      3 def view_events():
      4     df = spark.readStream.table("olympics.silver.events")
      5     return df

File /databricks/spark/python/pysparkpipelines/api.py:826, in view(query_function, name, comment, spark_conf)
    797 def view(
    798     query_function: Optional[Union[QueryFunction, DatasetOrExpectationDecoratorResult]] = None,
    799     name: Optional[str] = None,
   (...)
    807     DatasetOrExpectationDecoratorResult,
    808 ]:
    809     """
    810     (Return a) decorator to define a view in the pipeline and mark a function as the view's query
    811     function.
   (...)
    824         confs set for the pipeline or on the cluster.
    825     """
--> 826     return _view(
    827         decorator_name="view",
    828         query_functi

#CDC - DLT Tables

In [0]:
@dlt.view

def view_athletes():
    df = spark.readStream.table("olympics.silver.athletes")
    return df

In [0]:
dlt.create_streaming_table("athletes")

In [0]:
dlt.apply_changes(

    target = "athletes",
    source = "LIVE.view_athletes",
    keys=["athlete_id"],
    sequence_by = col("height"),
    stored_as_scd_type= 1

)